In [1]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    trim_messages,
)

In [2]:
from langchain_aws import ChatBedrockConverse
llm=ChatBedrockConverse(model='cohere.command-r-plus-v1:0', #amazon.nova-lite-v1:0
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)
llm.invoke("Hi").content

'Hello! How can I help you today?'

In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory 

#Dictionary to store session-based message histories
chat_history = {}
#Function to retrieve or create chat history for a given session ID
def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    history = chat_history.get(session_id) 
    if history is None: 
        history=InMemoryChatMessageHistory() 
        chat_history[session_id]=history 
        return history 

    trimmed_messages=trim_messages( 
        messages=history.messages, 
        token_counter=llm, 
        max_tokens=60, 
        strategy="last", 
        start_on="human", 
        allow_partial=False, 
        include_system=True 
    ) 
    history.messages=trimmed_messages 
    return history
        

In [4]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
#Define a prompt that expects "chat_history"
prompt = ChatPromptTemplate.from_messages([
	("system", "You are a helpful assistant."),
	MessagesPlaceholder(variable_name="chat_history"),
	("human", "{input}"),
	])

In [5]:
runnable = prompt | llm 

chain = RunnableWithMessageHistory(
	runnable,
	get_session_history,
	input_messages_key="input",
	history_messages_key="chat_history"
	)

In [6]:
# Simulate conversation with session ID "test_session"
# First message: Starts a conversation
response1 = chain.invoke({"input":"Hi there!"}, config={"configurable": {"session_id": "test_session"}})
print("Response 1:", response1)
print("_____"*20)
# Second message: Continues the conversation using the same session memory
response2 = chain.invoke(
    {"input":"I live in England and I love to eat dosa"},
    config={"configurable": {"session_id": "test_session"}}
)
print("Response 2:", response2)

Response 1: content='Hello! How can I help you today?' additional_kwargs={} response_metadata={'ResponseMetadata': {'RequestId': '09c49ca7-6aa4-4576-be07-45430981800e', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Mon, 20 Apr 2026 06:41:39 GMT', 'content-type': 'application/json', 'content-length': '232', 'connection': 'keep-alive', 'x-amzn-requestid': '09c49ca7-6aa4-4576-be07-45430981800e'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [420]}, 'model_provider': 'bedrock_converse', 'model_name': 'cohere.command-r-plus-v1:0'} id='lc_run--019da99f-8c52-77e0-a8eb-6f552af13213-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}}
____________________________________________________________________________________________________


/home/labuser/.local/lib/python3.10/site-packages/langchain_core/language_models/base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


Response 2: content="That's great! Dosa is a popular dish in South India, and it's also enjoyed by many people around the world. It's a thin, crispy crepe made from fermented rice and lentil batter, typically served with a variety of savory fillings and dips.\n\nIf you're looking for a place to eat dosa in England, there are several Indian restaurants that serve this dish. You can also try making it at home - there are many recipes available online, and it's a fun and rewarding cooking project." additional_kwargs={} response_metadata={'ResponseMetadata': {'RequestId': '436c7d2e-7b48-4890-88db-a37ebc473bca', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Mon, 20 Apr 2026 06:41:43 GMT', 'content-type': 'application/json', 'content-length': '683', 'connection': 'keep-alive', 'x-amzn-requestid': '436c7d2e-7b48-4890-88db-a37ebc473bca'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [2782]}, 'model_provider': 'bedrock_converse', 'model_name': 'cohere.command-r-plus

In [7]:
session_id="test_session"

In [8]:
print("Final Trimmed Buffer Window:")
for msg in chat_history[session_id].messages:
    print(f"{msg.type.capitalize()}: {msg.content}")

Final Trimmed Buffer Window:
Human: Hi there!
Ai: Hello! How can I help you today?
Human: I live in England and I love to eat dosa
Ai: That's great! Dosa is a popular dish in South India, and it's also enjoyed by many people around the world. It's a thin, crispy crepe made from fermented rice and lentil batter, typically served with a variety of savory fillings and dips.

If you're looking for a place to eat dosa in England, there are several Indian restaurants that serve this dish. You can also try making it at home - there are many recipes available online, and it's a fun and rewarding cooking project.


In [9]:
response3 = chain.invoke({"input":"My favorite player is virat kohli"}, config={"configurable": {"session_id": "test_session"}})
print("Response 3:", response3)

Response 3: content='Virat Kohli is a fantastic cricket player! He is known for his exceptional batting skills and has achieved numerous milestones in his career. He has a large fan following and is considered one of the best cricket players in the world.' additional_kwargs={} response_metadata={'ResponseMetadata': {'RequestId': 'e94546e8-7c73-44f3-b011-036074652925', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Mon, 20 Apr 2026 06:41:59 GMT', 'content-type': 'application/json', 'content-length': '437', 'connection': 'keep-alive', 'x-amzn-requestid': 'e94546e8-7c73-44f3-b011-036074652925'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [1300]}, 'model_provider': 'bedrock_converse', 'model_name': 'cohere.command-r-plus-v1:0'} id='lc_run--019da99f-d8bb-7770-b432-ad8815ddc7a9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 14, 'output_tokens': 45, 'total_tokens': 59, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}}


In [10]:
print("Final Trimmed Buffer Window:")
for msg in chat_history[session_id].messages:
    print(f"{msg.type.capitalize()}: {msg.content}")

Final Trimmed Buffer Window:
Human: My favorite player is virat kohli
Ai: Virat Kohli is a fantastic cricket player! He is known for his exceptional batting skills and has achieved numerous milestones in his career. He has a large fan following and is considered one of the best cricket players in the world.


In [11]:
response4 = chain.invoke({"input":"Who is my favorite player?"}, config={"configurable": {"session_id": "test_session"}})
print("Response 4:", response4)

Response 4: content='Your favorite player is Virat Kohli.' additional_kwargs={} response_metadata={'ResponseMetadata': {'RequestId': '8ca4b272-595b-40f0-b689-46fe20a2a1bb', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Mon, 20 Apr 2026 06:42:42 GMT', 'content-type': 'application/json', 'content-length': '237', 'connection': 'keep-alive', 'x-amzn-requestid': '8ca4b272-595b-40f0-b689-46fe20a2a1bb'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [476]}, 'model_provider': 'bedrock_converse', 'model_name': 'cohere.command-r-plus-v1:0'} id='lc_run--019da9a0-827b-7c03-8cbc-b8eb4a5770f8-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 65, 'output_tokens': 8, 'total_tokens': 73, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}}


In [12]:
print("Final Trimmed Buffer Window:")
for msg in chat_history[session_id].messages:
    print(f"{msg.type.capitalize()}: {msg.content}")

Final Trimmed Buffer Window:
Human: My favorite player is virat kohli
Ai: Virat Kohli is a fantastic cricket player! He is known for his exceptional batting skills and has achieved numerous milestones in his career. He has a large fan following and is considered one of the best cricket players in the world.
Human: Who is my favorite player?
Ai: Your favorite player is Virat Kohli.


In [12]:
response5 = chain.invoke({"input":"Where do I live?"}, config={"configurable": {"session_id": "test_session"}})
print("Response 5:", response5)

Response 5: content="I don't know where you live. Can you tell me more about your city or town?" additional_kwargs={} response_metadata={'ResponseMetadata': {'RequestId': '90c601df-4015-441f-9451-a833ef60b42a', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Mon, 20 Apr 2026 02:05:54 GMT', 'content-type': 'application/json', 'content-length': '276', 'connection': 'keep-alive', 'x-amzn-requestid': '90c601df-4015-441f-9451-a833ef60b42a'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [681]}, 'model_provider': 'bedrock_converse', 'model_name': 'cohere.command-r-plus-v1:0'} id='lc_run--019da8a3-1782-7f10-89bf-c6519979374f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 41, 'output_tokens': 19, 'total_tokens': 60, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}}
